# Laboratório de dois corpos a baixa energia
### Do potencial ao estado ligado, para qualquer potencial

**Pedro Henrique Gesualdo Modesto** — IFSC/USP
Orientador: Lucas Madeira · Coorientadora: Patrícia C. M. Castilho

> ### ⚠ Rode com **Run All** (ou Cell → Run All)
>
> A **primeira célula de código carrega todas as ferramentas** que o resto do
> caderno usa. Rodar uma célula do meio sem ter rodado ela dá
> `NameError: name 'medir' is not defined`. Depois que ela rodar uma vez,
> qualquer célula pode ser executada isoladamente, em qualquer ordem.
>
> Tempo total: **4 a 6 minutos** (as três sintonias do Lennard-Jones são o grosso).

---

## O que este documento faz

Duas partículas interagem por um potencial. A baixa energia, **a forma detalhada
do potencial não importa** — só dois números importam:

| Símbolo | Nome completo | O que é |
|---|---|---|
| `a` | comprimento de espalhamento | onde a reta assintótica da função de onda cruza o zero |
| `r0` | alcance efetivo | primeira correção de tamanho finito do potencial |

A expansão de alcance efetivo diz isso de forma exata:

$$ k\cot\delta_0(k) \;=\; -\frac{1}{a} \;+\; \frac{1}{2}\,r_0\,k^2 \;+\; O(k^4) $$

**A pergunta central deste caderno:** se dois potenciais completamente
diferentes forem *sintonizados* para dar o mesmo `a` e o mesmo `r0`, eles
preveem a mesma energia de estado ligado? Onde essa promessa vale, e onde
ela quebra?

## O caminho, em cinco passos

1. **Os potenciais** — quatro formas funcionais diferentes
2. **A medida** — extrair `a` e `r0` de qualquer um deles
3. **A sintonia automática** — dado o alvo (`a`, `r0`), achar os parâmetros
4. **O estado ligado** — resolver a energia e comparar com as fórmulas universais
5. **As varreduras** — varrer parâmetros em 1D e 2D e ver a estrutura aparecer

---

## Legenda de TODOS os símbolos usados

Nenhum símbolo solto neste caderno. Esta é a lista completa:

| No código | Na física | Significado |
|---|---|---|
| `comprimento_espalhamento` | $a$ | comprimento de espalhamento (fm) |
| `alcance_efetivo` | $r_0$ | alcance efetivo (fm) |
| `intensidade` | $v$ ou $C_6$ | quão fundo é o poço |
| `escala_inversa` | $\mu$ | inverso do alcance: $R = 1/\mu$ (fm⁻¹) |
| `alcance_potencial` | $R$ | onde o potencial deixa de agir (fm) |
| `energia_ligado` | $E$ | energia do estado ligado (negativa) |
| `numero_de_nos` | $n$ | quantos zeros a função de onda tem — conta os estados ligados |
| `funcao_onda_radial` | $u(r)$ | função de onda radial reduzida, $u = r\,\psi$ |
| `h2_sobre_2mu` | $\hbar^2/2\mu_{red}$ | converte unidades do código para MeV ou Kelvin |

**Convenção de unidades do laboratório:** $\hbar = \mu_{red} = 1$, comprimentos
em fm. Com isso a equação radial fica $u'' = 2\,(V - E)\,u$. Para voltar a
unidades físicas: $E_{físico} = 2\,(\hbar^2/2\mu_{red})\,E_{código}$.

In [ ]:
# ============================================================================
#  FERRAMENTAS  —  RODE ESTA CÉLULA PRIMEIRO
# ============================================================================
#  Tudo que o caderno usa está definido aqui: os imports, as cores, e as cinco
#  funções. Ficam juntas de propósito — assim qualquer célula lá de baixo roda
#  sozinha depois que esta rodar uma vez, sem depender da ordem.
#
#     medir(potencial)              -> (a, r0, número de nós)
#     sintonizar(potencial, caso)   -> parâmetros que reproduzem o alvo (a, r0)
#     raio_de_casamento(potencial)  -> onde o potencial já pode ser ignorado
#     energia_ligado(potencial)     -> energia do estado ligado fundamental
#     bancada(sistema, unidade)     -> sintoniza os 4 e compara com o experimento
# ============================================================================
# Tudo que é motor de cálculo já está testado em src/ (71 testes passando).
# Este caderno NÃO reimplementa nada: ele usa e compara. É de propósito —
# código duplicado é código que diverge.
import sys, math, itertools
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# a raiz do repositório é a pasta que contém src/
RAIZ = Path.cwd()
while not (RAIZ / "src").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.dois_corpos import espalhamento, ajuste, analitico
from src.dois_corpos.potenciais import (PocoEsferico, PoschlTeller,
                                        Gaussiano, LennardJones, FABRICAS)
from src.comum import solvers
from src.literatura.tabelas_artigo import TABELA1, TABELA2, TABELA3, TABELA4

plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.3,
                     "figure.autolayout": True})

# Uma cor por potencial, usada em TODO o caderno. Cor constante é o que
# permite ler seis gráficos diferentes sem consultar legenda toda hora.
COR = {"poco": "#1f77b4", "mpt": "#d62728", "gauss": "#2ca02c", "lj": "#9467bd"}
NOME_LONGO = {"poco": "Poço esférico", "mpt": "Pöschl-Teller modificado",
              "gauss": "Gaussiano", "lj": "Lennard-Jones"}


print(f"raiz do repositório: {RAIZ}")


def medir(potencial, passo_radial=1e-3, metodo="numerov"):
    """Devolve (comprimento_espalhamento, alcance_efetivo, numero_de_nos).

    Envelope fino em cima de espalhamento.calcular() só para dar nomes
    completos ao que sai — o cálculo é o do módulo testado.
    """
    r = espalhamento.calcular(potencial, dr=passo_radial, metodo=metodo)
    return r.a, r.r0, r.nos


# Passo radial POR POTENCIAL. Não é capricho: o Lennard-Jones tem cauda 1/r^6,
# então o alcance numérico dele (onde |V| < 1e-15) chega a centenas de fm. Com
# passo 2e-3 são 200 mil pontos POR AVALIAÇÃO, e a sintonia faz centenas de
# avaliações. Com 2e-2 o resultado muda na 4ª casa decimal e o cálculo cai de
# minutos para segundos. Medir o custo antes de escolher o passo faz parte.
PASSO_RADIAL = {"poco": 5e-3, "mpt": 5e-3, "gauss": 5e-3, "lj": 2e-2}


def sintonizar(nome_potencial, caso, passo_radial=None, verbose=True):
    """Acha os parâmetros do potencial que reproduzem o alvo (a, r0) do caso.

    caso : "nn" (nêutron-nêutron), "unitario" ou "deuteron" — Tabela 2.
    Chute inicial: os valores publicados (Tabelas 3 e 4). Isso é honesto —
    a comparação é para VALIDAR o método, e um bom chute é o que se usaria
    de verdade num alvo novo (extrapolando de um caso vizinho).
    """
    if passo_radial is None:
        passo_radial = PASSO_RADIAL[nome_potencial]
    alvo = TABELA2[caso]
    tabela = TABELA3 if nome_potencial != "lj" else TABELA4
    chute = tabela[(caso, nome_potencial)]

    resultado = ajuste.ajustar(
        nome_potencial,
        a_alvo=alvo["a"], r0_alvo=alvo["r0"],
        p1_ini=chute["p1"], p2_ini=chute["p2"],
        dr=passo_radial, nos_alvo=alvo["nos"])

    if verbose:
        marca = "ok" if resultado.convergiu else "NAO CONVERGIU"
        print(f"  {NOME_LONGO[nome_potencial]:<26} "
              f"p1={resultado.p1:>12.6f}  p2={resultado.p2:>12.6f}  "
              f"a={resultado.a:>11.4g}  r0={resultado.r0:>7.4f}  "
              f"nós={resultado.nos}  [{marca}, {resultado.iteracoes} it]")
    return resultado


def raio_de_casamento(potencial, tolerancia=1e-8):
    """Menor raio a partir do qual |V(r)| < tolerância — dali em diante a
    solução já é a exponencial livre e não precisamos integrar."""
    r = np.linspace(max(potencial.r_min, 1e-6), min(potencial.R, 1e4), 20000)
    ainda_age = np.abs(potencial.V(r)) >= tolerancia
    return float(r[ainda_age][-1]) if ainda_age.any() else float(r[0])


def _integra_ate(potencial, energia, passo, raio_casamento):
    """Integra u'' = 2(V - E)u de r_min até o raio de casamento (Verlet/ordem 2).

    O re-escalonamento por 1e200 evita overflow. O laboratório já foi mordido
    por isso: contar nós DEPOIS do re-escalonamento apaga os nós iniciais por
    underflow (Achado registrado no HISTORICO). Por isso conta-se na hora.
    """
    r = np.arange(potencial.r_min, raio_casamento + 2*passo, passo)
    W = 2.0 * (potencial.V(r) - energia)
    funcao_onda_radial = np.zeros(r.size)
    funcao_onda_radial[1] = passo               # u(0)=0 e inclinação arbitrária
    numero_de_nos = 0
    u = funcao_onda_radial
    for i in range(1, r.size - 1):
        u[i+1] = 2*u[i] - u[i-1] + passo*passo*W[i]*u[i]
        if (u[i+1] < 0) != (u[i] < 0) and u[i] != 0:
            numero_de_nos += 1
        if abs(u[i+1]) > 1e200:
            u[:i+2] /= 1e200
    return u, r, numero_de_nos


def energia_ligado(potencial, passo=None, iteracoes=90, n_varredura=250):
    """Energia do estado ligado FUNDAMENTAL, em unidades do código (fm⁻²).

    Devolve NaN se o potencial não liga nada — que é uma resposta legítima,
    não um erro.
    """
    Rc = raio_de_casamento(potencial)
    if passo is None:
        passo = (potencial.r_min/60.0) if potencial.r_min > 0 else Rc/2500.0
    fundo_do_poco = float(np.min(potencial.V(np.arange(potencial.r_min, Rc, passo))))

    def g(energia):
        u, r, _ = _integra_ate(potencial, energia, passo, Rc)
        i = len(r) - 2
        derivada = (u[i+1] - u[i-1]) / (2*passo)
        escala = np.max(np.abs(u)) or 1.0        # só para não estourar
        return (derivada + math.sqrt(-2.0*energia)*u[i]) / escala

    energias = -np.logspace(math.log10(abs(fundo_do_poco)), -12, n_varredura)
    valores = [g(E) for E in energias]
    for k in range(len(energias) - 1):
        if valores[k]*valores[k+1] < 0:
            baixo, alto = energias[k], energias[k+1]
            break
    else:
        return float("nan")
    for _ in range(iteracoes):
        meio = 0.5*(baixo + alto)
        if g(baixo)*g(meio) <= 0: alto = meio
        else: baixo = meio
    return 0.5*(baixo + alto)


CHUTE_BASE = {"poco": (1.7806, 0.48415), "mpt": (1.44397, 0.853766),
              "gauss": (1.93585, 0.652551), "lj": (7.8433, 1.27426)}

def bancada(chave_tabela, unidade, potenciais=("poco", "mpt", "gauss", "lj")):
    """Sintoniza no alvo da Tabela 1 e calcula a energia com cada potencial."""
    alvo = TABELA1[chave_tabela]
    h2_sobre_2mu = -alvo["E_zr_ref"] * alvo["a"]**2      # inferido, ver nota
    fator = 2.0 * h2_sobre_2mu                            # E_fisico = fator * E_codigo
    lam = alvo["r0"] / 1.7436                             # escala vs. dêuteron

    print(f"\n{'='*88}\n {chave_tabela.upper()}   alvo: a = {alvo['a']} , "
          f"r0 = {alvo['r0']}    (ħ²/2μ = {h2_sobre_2mu:.3f} {unidade}·comp²)")
    print(f" EXPERIMENTO: E = {alvo['E_ref']:.5g} {unidade}\n{'-'*88}")
    E_zr = analitico.energia_zr(alvo["a"], h2_sobre_2mu)
    E_fr = analitico.energia_fr(alvo["a"], alvo["r0"], h2_sobre_2mu)
    print(f" {'fórmula de alcance ZERO':<34}{E_zr:>14.5g} {unidade}"
          f"{100*(E_zr/alvo['E_ref']-1):>+10.2f}%")
    print(f" {'fórmula de alcance FINITO':<34}{E_fr:>14.5g} {unidade}"
          f"{100*(E_fr/alvo['E_ref']-1):>+10.2f}%")
    print(f"{'-'*88}")

    saida = {}
    for nome in potenciais:
        p1, p2 = CHUTE_BASE[nome]
        p2_ini = p2 * (lam**6 if nome == "lj" else 1.0/lam)  # C12 escala com λ^6
        passo = (2e-2 if nome == "lj" else 5e-3) * lam
        s = ajuste.ajustar(nome, alvo["a"], alvo["r0"], p1, p2_ini,
                           dr=passo, nos_alvo=1)
        E = fator * energia_ligado(FABRICAS[nome](s.p1, s.p2))
        saida[nome] = {"sintonia": s, "E": E}
        print(f" {NOME_LONGO[nome]:<34}{E:>14.5g} {unidade}"
              f"{100*(E/alvo['E_ref']-1):>+10.2f}%   "
              f"(p1={s.p1:.5f}, p2={s.p2:.6f})")
    saida["_meta"] = {"alvo": alvo, "unidade": unidade, "fator": fator,
                      "E_zr": E_zr, "E_fr": E_fr}
    return saida

print("ferramentas carregadas: medir, sintonizar, raio_de_casamento, energia_ligado, bancada")

---
# 1. Os quatro potenciais

São quatro formas funcionais deliberadamente diferentes entre si. A escolha
não é decorativa — cada uma testa uma hipótese distinta:

| Potencial | Forma | O que ele testa |
|---|---|---|
| Poço esférico | $-v\mu^2$ para $r<1/\mu$ | **descontinuidade**: borda abrupta |
| Pöschl-Teller modificado | $-v\mu^2/\cosh^2(\mu r)$ | **suave e solúvel**: tem solução analítica |
| Gaussiano | $-v\mu^2 e^{-\mu^2r^2}$ | **cauda que morre rápido** (mais que exponencial) |
| Lennard-Jones | $\tfrac12(C_{12}/r^{12} - C_6/r^6)$ | **caroço duro**: $V\to+\infty$ em $r\to0$, cauda de van der Waals |

O Lennard-Jones é o mais realista para átomos frios — é o potencial de
verdade entre dois átomos neutros. Também é o mais difícil numericamente,
justamente por causa do caroço.

> **Atenção à convenção (Achado nº 1 do laboratório):** o fator $\tfrac12$ no
> Lennard-Jones. Metade da literatura escreve $C_{12}/r^{12}-C_6/r^6$ sem ele.
> Usar a convenção errada dá um $C_6$ com fator 2 de diferença e o resultado
> *parece* certo. Aqui usamos a convenção com o $\tfrac12$, e ela está
> registrada em `referencias/CONVENCOES.md`.

In [ ]:
# ---------------------------------------------------------------------------
#  Construímos os quatro no caso "unitariedade" da Tabela 3 do artigo.
#  Unitariedade = |a| -> infinito. É o ponto mais interessante da física de
#  átomos frios: o potencial está EXATAMENTE no limiar de ligar um estado.
# ---------------------------------------------------------------------------
CASO_DEMO = "unitario"

potenciais_demo = {}
for nome in ("poco", "mpt", "gauss"):
    par = TABELA3[(CASO_DEMO, nome)]
    potenciais_demo[nome] = FABRICAS[nome](par["p1"], par["p2"])
par_lj = TABELA4[(CASO_DEMO, "lj")]
potenciais_demo["lj"] = FABRICAS["lj"](par_lj["p1"], par_lj["p2"])

fig, (esq, dir_) = plt.subplots(1, 2, figsize=(11, 4))

# --- painel da esquerda: os três potenciais suaves, escala linear ----------
raio = np.linspace(1e-4, 4.0, 800)
for nome in ("poco", "mpt", "gauss"):
    esq.plot(raio, potenciais_demo[nome].V(raio), color=COR[nome], lw=1.8,
             label=NOME_LONGO[nome])
esq.axhline(0, color="k", lw=0.6)
esq.set(xlabel="raio r (fm)", ylabel="potencial V(r) (fm⁻²)",
        title=f"Três potenciais sintonizados no mesmo alvo\n(caso: {CASO_DEMO})")
esq.legend(fontsize=8)

# --- painel da direita: o Lennard-Jones, que precisa de escala própria ------
# O caroço vai a +1e10; sem escala log não se vê nada. E o mínimo dele fica
# em r ~ 0.06 fm com esses parâmetros: o LJ sintonizado na unitariedade é um
# potencial de MUITO curto alcance.
raio_lj = np.logspace(-2.0, 0.5, 800)
V_lj = potenciais_demo["lj"].V(raio_lj)
dir_.plot(raio_lj, V_lj, color=COR["lj"], lw=1.8, label=NOME_LONGO["lj"])
dir_.axhline(0, color="k", lw=0.6)
dir_.axvline(potenciais_demo["lj"].r_min, color="gray", ls=":", lw=1.2,
             label=f"corte do caroço r_min = {potenciais_demo['lj'].r_min:.3g} fm")
dir_.set(xscale="log", yscale="symlog", xlabel="raio r (fm)  [escala log]",
         ylabel="potencial V(r) (fm⁻²)  [escala symlog]",
         title="Lennard-Jones: caroço duro + cauda de van der Waals")
dir_.legend(fontsize=8)
plt.show()

# A tabela abaixo deixa explícito o que "alcance" significa para cada um:
# o código define R como o ponto onde |V(R)| cai abaixo de 1e-15.
print(f"{'potencial':<26} {'parâmetros':<34} {'alcance R (fm)':>14}")
print("-" * 76)
for nome, pot in potenciais_demo.items():
    par_txt = ", ".join(f"{k}={v:.6g}" for k, v in pot.parametros().items())
    print(f"{NOME_LONGO[nome]:<26} {par_txt:<34} {pot.R:>14.4f}")

---
# 2. A medida: extrair `a` e `r0` de qualquer potencial

**Uma função só, e ela serve para os quatro potenciais.** Isso é o ponto:
o motor não sabe qual potencial recebeu, só sabe integrar.

### Como funciona, em três linhas de física

1. Integra $u'' = 2V(r)\,u$ de $r=0$ até depois do alcance, com $u(0)=0$.
2. Fora do alcance $V=0$, logo $u$ é uma **reta**. O comprimento de
   espalhamento é onde essa reta cruza o zero:
   $\;a = R - u(R)/u'(R)$.
3. O alcance efetivo sai de uma integral que mede o quanto a solução
   verdadeira difere da reta:
   $\;r_0 = 2\int_0^\infty\!\left[g_0(r)^2 - u(r)^2\right]dr$, com $g_0 = 1 - r/a$.

### Três decisões numéricas que importam

- **A grade é alinhada à borda do poço** (`solvers.grade`). Sem isso o erro
  do poço esférico não cai com a ordem esperada — a descontinuidade fica
  entre dois pontos e o método não a enxerga.
- **Numerov (ordem 4) para potenciais suaves.** Em potencial descontínuo ele
  degrada para ordem 1 — é o *Achado nº 2* do laboratório, registrado no
  `HISTORICO.md`. Não é bug: é a hipótese de suavidade quebrando.
- **A integral usa Simpson**, que é o valor oficial; o trapézio fica
  guardado como conferência barata.

In [ ]:
# ---------------------------------------------------------------------------
#  Aplicando aos quatro potenciais da seção anterior.
#  O alvo (Tabela 2 do artigo) para o caso "unitario" é |a| -> infinito e
#  r0 = 1.0 fm. Um |a| de ordem 1e5 fm JÁ É infinito para fins práticos:
#  a escala física do problema é 1 fm.
# ---------------------------------------------------------------------------
alvo = TABELA2[CASO_DEMO]
print(f"ALVO ({CASO_DEMO}):  a = {alvo['a']},  r0 = {alvo['r0']} fm,  "
      f"nós esperados = {alvo['nos']}\n")
print(f"{'potencial':<26} {'a (fm)':>14} {'r0 (fm)':>10} {'nós':>5}   "
      f"{'erro em r0':>11}")
print("-" * 74)
for nome, pot in potenciais_demo.items():
    comprimento_espalhamento, alcance_efetivo, numero_de_nos = medir(pot)
    erro = alcance_efetivo - alvo["r0"]
    print(f"{NOME_LONGO[nome]:<26} {comprimento_espalhamento:>14.4g} "
          f"{alcance_efetivo:>10.4f} {numero_de_nos:>5}   {erro:>+11.2e}")

### Leitura do resultado

Os quatro têm `r0` batendo em 1.0 fm e `|a|` gigante — estão todos no mesmo
ponto do espaço $(a, r_0)$, apesar de terem formas completamente diferentes.
**É exatamente isso que "universalidade" quer dizer.**

O sinal de `a` alterna entre eles e isso é irrelevante: na unitariedade
$1/a = 0$, e $+\infty$ e $-\infty$ são o mesmo ponto. O que tem significado
físico é $1/a$, não $a$.

---
# 3. A sintonia automática

**Este é o coração do caderno.** O problema inverso:

> Dado um alvo $(a, r_0)$, ache os parâmetros de *qualquer* potencial que
> reproduzam esse alvo.

### Por que é difícil

Os dois parâmetros mexem nas duas grandezas ao mesmo tempo. Aumentar a
profundidade muda `a` **e** `r0`. É um sistema 2×2 não-linear, e a função
$a(\text{intensidade})$ tem **polos**: ela salta de $+\infty$ para $-\infty$
toda vez que um novo estado ligado aparece.

### A estratégia que funciona: dois laços aninhados em $1/a$

```
laço externo:  ajusta a ESCALA (mu ou C12)  ->  acerta r0
    laço interno:  ajusta a INTENSIDADE (v ou C6)  ->  acerta 1/a
```

**A decisão que faz tudo funcionar: procurar a raiz de $1/a - 1/a_{alvo}$, e
não de $a - a_{alvo}$.** O motivo é geométrico: $a$ tem polos, $1/a$ é suave
e cruza o zero. Um método de bisseção morre em cima de um polo e prospera
em cima de um zero. Como bônus, unitariedade ($a=\infty$) vira simplesmente
$1/a = 0$ — não precisa de caso especial nenhum.

A raiz é achada pelo **método de Illinois** (variante da falsa posição que
não sofre da estagnação de um dos extremos), já implementado em
`src/dois_corpos/ajuste.py`.

### O portão obrigatório: contar os nós

Cada polo de $a$ é um estado ligado novo. Duas soluções podem ter o mesmo
$(a, r_0)$ e um número de nós diferente — e aí **não são o mesmo estado
físico**. Por isso toda sintonia termina conferindo `numero_de_nos`. Sem esse
portão, o ajuste converge para a resposta errada em silêncio.

In [ ]:
# ---------------------------------------------------------------------------
#  Reproduzindo as Tabelas 3 e 4 do artigo INTEIRAS: 3 casos x 4 potenciais.
#  12 sintonias independentes. Cada uma parte dos parâmetros publicados e
#  tem que voltar neles.
# ---------------------------------------------------------------------------
CASOS = ("nn", "unitario", "deuteron")
ROTULO_CASO = {"nn": "Nêutron-nêutron", "unitario": "Unitariedade",
               "deuteron": "Dêuteron"}

sintonias = {}
for caso in CASOS:
    alvo = TABELA2[caso]
    print(f"\n=== {ROTULO_CASO[caso]}:  a = {alvo['a']} fm, "
          f"r0 = {alvo['r0']} fm, nós = {alvo['nos']} ===")
    for nome in ("poco", "mpt", "gauss", "lj"):
        sintonias[(caso, nome)] = sintonizar(nome, caso)

### Sintonia vs. valores publicados

A tabela abaixo é o teste de validação: partimos dos parâmetros publicados,
rodamos a sintonia, e ela tem que **voltar neles**. Diferença grande aqui
significaria que a nossa definição de `a` ou de `r0` difere da do artigo —
o tipo de erro de convenção que é invisível se você não fizer esta conferência.

In [ ]:
print(f"{'caso':<16}{'potencial':<26}{'p1 (nosso)':>12}{'p1 (art.)':>11}"
      f"{'p2 (nosso)':>12}{'p2 (art.)':>11}{'maior dif.':>12}")
print("-" * 100)
for caso in CASOS:
    for nome in ("poco", "mpt", "gauss", "lj"):
        r = sintonias[(caso, nome)]
        pub = (TABELA3 if nome != "lj" else TABELA4)[(caso, nome)]
        # diferença relativa no parâmetro que mais se afastou
        dif = max(abs(r.p1/pub["p1"] - 1), abs(r.p2/pub["p2"] - 1))
        print(f"{ROTULO_CASO[caso]:<16}{NOME_LONGO[nome]:<26}"
              f"{r.p1:>12.6f}{pub['p1']:>11.4f}{r.p2:>12.6f}{pub['p2']:>11.4f}"
              f"{100*dif:>11.2f}%")

### O quadro completo da sintonia, em dois gráficos

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(13, 3.8))

# --- (a) os potenciais sintonizados no MESMO alvo, lado a lado ---------------
# Três formas diferentes que produzem exatamente o mesmo par (a, r0).
raio = np.linspace(1e-4, 6.0, 700)
for nome in ("poco", "mpt", "gauss"):
    r = sintonias[("deuteron", nome)]
    eixos[0].plot(raio, FABRICAS[nome](r.p1, r.p2).V(raio),
                  color=COR[nome], lw=1.8, label=NOME_LONGO[nome])
eixos[0].axhline(0, color="k", lw=0.6)
eixos[0].set(xlabel="raio r (fm)", ylabel="V(r) (fm⁻²)", xlim=(0, 6),
             title="Sintonizados no dêuteron\na = 5.4 fm, r₀ = 1.7 fm")
eixos[0].legend(fontsize=7)

# --- (b) profundidade necessária, por caso ----------------------------------
# Quanto mais fundo o poço tem que ser? Depende do alvo E da forma.
larg = 0.25
for k, nome in enumerate(("poco", "mpt", "gauss")):
    alturas = [sintonias[(c, nome)].p1 for c in CASOS]
    eixos[1].bar(np.arange(3) + (k-1)*larg, alturas, larg,
                 color=COR[nome], label=NOME_LONGO[nome])
eixos[1].set_xticks(range(3), [ROTULO_CASO[c] for c in CASOS], fontsize=8)
eixos[1].set(ylabel="intensidade v (adimensional)",
             title="Profundidade necessária\npara atingir cada alvo")
eixos[1].legend(fontsize=7)

# --- (c) o plano (a, r0): onde cada alvo mora -------------------------------
# Cada caso é UM PONTO neste plano. Todos os potenciais sintonizados caem
# exatamente em cima dele — é a definição operacional de universalidade.
for caso in CASOS:
    alvo = TABELA2[caso]
    x = 0.0 if math.isinf(alvo["a"]) else 1.0/alvo["a"]
    eixos[2].scatter(x, alvo["r0"], s=120, marker="*", zorder=5,
                     label=f"{ROTULO_CASO[caso]}  (nós={alvo['nos']})")
    for nome in ("poco", "mpt", "gauss", "lj"):
        r = sintonias[(caso, nome)]
        eixos[2].scatter(1.0/r.a, r.r0, s=26, color=COR[nome], zorder=6)
eixos[2].axvline(0, color="gray", ls=":", lw=1)
eixos[2].text(0.002, 2.9, "unitariedade\n1/a = 0", fontsize=7, color="gray")
eixos[2].set(xlabel="1/a  (fm⁻¹)   ← é 1/a que tem sentido físico, não a",
             ylabel="alcance efetivo r₀ (fm)",
             title="O plano (1/a, r₀)\nestrela = alvo · pontos = os 4 potenciais")
eixos[2].legend(fontsize=7, loc="lower left")
plt.show()

---
# 4. O estado ligado

Aqui está a pergunta que interessa ao experimento: **sintonizados no mesmo
$(a, r_0)$, os quatro potenciais preveem a mesma energia de ligação?**

### O método: casamento da derivada logarítmica

Poderíamos "atirar" a solução até o infinito e procurar a energia em que ela
não explode. Não fazemos isso — para $a$ grande o estado é raso, o
comprimento de decaimento $1/\kappa$ fica gigantesco (no dímero de hélio,
~90 Å), e a grade fica impossível.

Em vez disso: fora do alcance $V=0$, então a solução **já é conhecida**,
$u \propto e^{-\kappa r}$ com $\kappa = \sqrt{-2E}$. Basta integrar até um raio
$R_c$ onde o potencial já morreu e exigir que as duas emendem:

$$ g(E) \;=\; u'(R_c) + \kappa(E)\,u(R_c) \;=\; 0 $$

**Por que essa forma e não $u'/u + \kappa$:** o dêuteron tem um nó, e $u(R_c)$
pode passar por zero. A razão explodiria num polo falso e a bisseção iria
atrás dele. Somando em vez de dividir, $g$ é suave e só zera nos estados
ligados de verdade.

**A varredura é logarítmica em $E$.** Estado ligado raso mora colado no zero
(o dímero de hélio está a $1.6\times10^{-3}$ K, com o poço a $\sim 1$ K de
profundidade). Uma grade linear passaria por cima dele sem enxergar.

### A prova: dois sistemas reais, quatro potenciais cada

**Dêuteron** (nêutron + próton, energia medida $-2.224$ MeV) e **dímero de
hélio-4** (dois átomos de $^4$He, $-1.62\times10^{-3}$ K). Sistemas separados
por **nove ordens de grandeza** em energia e por toda a física nuclear vs.
molecular.

Sintonizamos os quatro potenciais em cada um e comparamos com:

| Fórmula | Expressão | O que ignora |
|---|---|---|
| alcance zero | $E = -\dfrac{\hbar^2}{2\mu_{red}\,a^2}$ | o tamanho do potencial, inteiro |
| alcance finito | resolve $\kappa = \dfrac1a + \dfrac{r_0\kappa^2}{2}$ | os termos $O(k^4)$ |

> **Nota de unidades.** $\hbar^2/2\mu_{red}$ não está tabelado: nós o
> *inferimos* da própria Tabela 1, invertendo a fórmula de alcance zero
> ($\hbar^2/2\mu_{red} = -E_{zr}\,a^2$). Dá 41.46 MeV·fm² para o dêuteron
> (literatura: 41.47) e 12.09 K·Å² para o hélio (literatura: 12.12). Bater
> nas duas é a checagem de que a tabela é internamente consistente.

In [ ]:
# Chutes iniciais: reaproveitamos a solução do dêuteron e a REESCALAMOS.
# Isso é legítimo e é o que se faria com um alvo novo: o problema tem
# invariância de escala; multiplicar todos os comprimentos por lambda leva
# (a, r0) -> (lambda*a, lambda*r0). Basta dividir o parâmetro de escala.
resultado_deuteron = bancada("deuteron", "MeV")
# O Lennard-Jones fica de fora do hélio: com a = 90 Å a cauda 1/r^6 só se torna
# desprezível a milhares de Å, e o custo de uma grade uniforme explode. É uma
# limitação do método, não do potencial — e está anotada nas conclusões.
resultado_helio    = bancada("he4_dimer", "K",
                             potenciais=("poco", "mpt", "gauss"))

### Como ler isto

**A fórmula de alcance zero erra por 36% no dêuteron e 8.6% no hélio.**
A diferença entre os dois é `|a|/r0`: vale 3.1 no dêuteron e 11.3 no hélio.
Quanto maior essa razão, mais o sistema é "universal" e menos o tamanho do
potencial importa. **O dêuteron não é um sistema universal** — é o exemplo
didático clássico e, ironicamente, o que menos obedece.

**Incluir $r_0$ derruba o erro para menos de 1% nos dois casos.** Dois números
— e nada mais sobre a interação nuclear ou de van der Waals — bastam.

**Os quatro potenciais concordam dentro de ~1%.** O espalhamento residual
entre eles é a *dependência de forma*: os termos $O(k^4)$ da expansão, que
são a única coisa que ainda distingue um poço quadrado de um Lennard-Jones
depois que $a$ e $r_0$ foram fixados.

In [ ]:
# ---------------------------------------------------------------------------
#  A FIGURA: por que os potenciais concordam apesar de serem tão diferentes
# ---------------------------------------------------------------------------
fig, eixos = plt.subplots(1, 3, figsize=(13.5, 4))

# --- (a) as funções de onda: diferentes DENTRO, idênticas FORA ---------------
# Esta é a figura mais importante do caderno. É universalidade, visualmente.
for nome in ("poco", "mpt", "gauss"):
    s = resultado_deuteron[nome]["sintonia"]
    pot = FABRICAS[nome](s.p1, s.p2)
    res = espalhamento.calcular(pot, dr=1e-3)
    eixos[0].plot(res.r, res.u, color=COR[nome], lw=1.6, label=NOME_LONGO[nome])
a_alvo = TABELA1["deuteron"]["a"]
r_reta = np.linspace(0, 9, 50)
eixos[0].plot(r_reta, 1 - r_reta/a_alvo, "k--", lw=1.4,
              label=f"assíntota  g₀ = 1 − r/a")
eixos[0].scatter([a_alvo], [0], s=70, color="k", zorder=5)
eixos[0].annotate(f"a = {a_alvo} fm", (a_alvo, 0), (a_alvo-2.4, 0.33),
                  arrowprops=dict(arrowstyle="->", lw=0.9), fontsize=8)
eixos[0].axhline(0, color="k", lw=0.6); eixos[0].set_xlim(0, 9)
eixos[0].set(xlabel="raio r (fm)", ylabel="função de onda radial u(r)",
             title="(a) Diferentes DENTRO, idênticas FORA\nsintonizados no dêuteron")
eixos[0].legend(fontsize=7)

# --- (b) energia prevista por cada potencial vs. experimento ----------------
for k, (res, chave, unid) in enumerate(
        ((resultado_deuteron, "deuteron", "MeV"), (resultado_helio, "he4_dimer", "K"))):
    ax = eixos[1+k]
    meta = res["_meta"]; E_exp = meta["alvo"]["E_ref"]
    rotulos = ["alcance\nzero", "alcance\nfinito"] + \
              [NOME_LONGO[n].split()[0] for n in ("poco", "mpt", "gauss", "lj") if n in res]
    valores = [meta["E_zr"], meta["E_fr"]] + \
              [res[n]["E"] for n in ("poco", "mpt", "gauss", "lj") if n in res]
    cores = ["#888888", "#444444"] + [COR[n] for n in ("poco","mpt","gauss","lj") if n in res]
    desvios = [100*(v/E_exp - 1) for v in valores]
    ax.bar(range(len(valores)), desvios, color=cores)
    ax.axhline(0, color="k", lw=1.2)
    ax.axhspan(-1, 1, color="green", alpha=0.12)
    ax.text(0.02, 0.94, "faixa de ±1%", transform=ax.transAxes,
            fontsize=7, color="green", va="top")
    ax.set_xticks(range(len(valores)), rotulos, fontsize=7, rotation=30)
    ax.set(ylabel="desvio em relação ao experimento (%)",
           title=f"({'bc'[k]}) {chave}   |a|/r₀ = "
                 f"{abs(meta['alvo']['a'])/meta['alvo']['r0']:.1f}")
plt.show()

---
# 5. Varreduras de parâmetros

Até aqui resolvemos o problema **inverso** (dado o alvo, ache os parâmetros).
Agora o **direto**: varre os parâmetros e vê o que acontece. É aqui que a
estrutura do problema aparece — e é o que justifica cada escolha feita na
seção 3.

## 5.1 Varredura em profundidade: o nascimento dos estados ligados

Fixamos a escala ($\mu = 1$, alcance $R = 1$ fm) e varremos a intensidade $v$.

**O que esperar:** cada vez que o poço fica fundo o bastante para segurar mais
um estado, $a$ diverge — vai a $+\infty$, reaparece em $-\infty$. Esses polos
são **ressonâncias de espalhamento**. No poço esférico o primeiro acontece em
$v = \pi^2/8 = 1.2337$, valor exato conhecido, e o laboratório o guarda como
`analitico.V_LIMIAR_POCO`.

In [ ]:
intensidades = np.linspace(0.05, 12.0, 260)

varredura = {}
for nome in ("poco", "mpt", "gauss"):
    a_lista, r0_lista, nos_lista = [], [], []
    for v in intensidades:
        # escala_inversa = 1 -> alcance R = 1 fm para poço e gaussiana
        comprimento_espalhamento, alcance_efetivo, numero_de_nos = \
            medir(FABRICAS[nome](v, 1.0), passo_radial=2e-3)
        a_lista.append(comprimento_espalhamento)
        r0_lista.append(alcance_efetivo)
        nos_lista.append(numero_de_nos)
    varredura[nome] = (np.array(a_lista), np.array(r0_lista), np.array(nos_lista))

fig, eixos = plt.subplots(1, 3, figsize=(13.5, 4))

# --- (a) 1/a: suave, cruza o zero. É NESTA função que a sintonia procura raiz --
for nome in ("poco", "mpt", "gauss"):
    eixos[0].plot(intensidades, 1.0/varredura[nome][0], color=COR[nome],
                  lw=1.5, label=NOME_LONGO[nome])
eixos[0].axhline(0, color="k", lw=1.0)
eixos[0].axvline(analitico.V_LIMIAR_POCO, color=COR["poco"], ls=":", lw=1.2)
eixos[0].text(analitico.V_LIMIAR_POCO+0.15, 0.85,
              f"π²/8 = {analitico.V_LIMIAR_POCO:.4f}\n1º estado ligado do poço",
              fontsize=7, color=COR["poco"])
eixos[0].set(xlabel="intensidade v", ylabel="1/a  (fm⁻¹)",
             title="(a) 1/a é SUAVE e cruza o zero\n← por isso a sintonia usa 1/a")
eixos[0].legend(fontsize=7)

# --- (b) o mesmo em a: polos. Bisseção morre aqui. -------------------------
for nome in ("poco", "mpt", "gauss"):
    a = varredura[nome][0].copy()
    a[np.abs(a) > 60] = np.nan          # corta os ramos que fogem, só para ver
    eixos[1].plot(intensidades, a, color=COR[nome], lw=1.5)
eixos[1].axhline(0, color="k", lw=0.6)
eixos[1].set(xlabel="intensidade v", ylabel="a  (fm)", ylim=(-60, 60),
             title="(b) a tem POLOS\ncada salto = um estado ligado novo")

# --- (c) contagem de estados ligados: escada -------------------------------
for nome in ("poco", "mpt", "gauss"):
    eixos[2].step(intensidades, varredura[nome][2], where="post",
                  color=COR[nome], lw=1.6, label=NOME_LONGO[nome])
eixos[2].set(xlabel="intensidade v", ylabel="número de nós de u(r)",
             title="(c) Cada degrau = um estado ligado\né o portão que a sintonia confere")
eixos[2].legend(fontsize=7)
plt.show()

print(f"limiar exato do 1º estado ligado no poço (π²/8): "
      f"{analitico.V_LIMIAR_POCO:.6f}")
i_salto = int(np.argmax(varredura['poco'][2] > 0))
print(f"limiar medido pela varredura              : "
      f"{intensidades[i_salto]:.4f}  "
      f"(resolução da grade: {intensidades[1]-intensidades[0]:.4f})")

## 5.2 O alcance efetivo diverge nos mesmos pontos

$r_0$ tem polos exatamente onde $a$ tem zeros. Não é coincidência: perto de um
zero de $a$ a normalização da função de onda estoura. Isso significa que
**existem regiões do espaço de parâmetros onde nenhum alvo $(a, r_0)$
razoável é alcançável** — e a sintonia tem que ficar longe delas.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
for nome in ("poco", "mpt", "gauss"):
    a, r0, _ = varredura[nome]
    r0_corte = r0.copy(); r0_corte[np.abs(r0_corte) > 12] = np.nan
    ax.plot(intensidades, r0_corte, color=COR[nome], lw=1.5, label=NOME_LONGO[nome])
    # marca onde a passa por zero: é lá que r0 explode
    trocas = np.where(np.diff(np.sign(a)) != 0)[0]
    for t in trocas:
        if abs(a[t]) < 20:              # zero de a, não polo
            ax.axvline(intensidades[t], color=COR[nome], ls=":", lw=0.9, alpha=0.6)
ax.axhline(0, color="k", lw=0.6)
ax.set(xlabel="intensidade v", ylabel="alcance efetivo r₀ (fm)", ylim=(-12, 12),
       title="r₀ diverge onde a passa por ZERO (linhas pontilhadas)\n"
             "— regiões proibidas para a sintonia")
ax.legend(fontsize=8)
plt.show()

## 5.3 Varredura 2D: o mapa que a sintonia percorre

Agora os **dois** parâmetros ao mesmo tempo. As curvas de nível de $1/a$ e de
$r_0$ no plano (intensidade, escala) são duas famílias de curvas; **a solução
da sintonia é o cruzamento delas**. Ver isso desenhado explica de uma vez por
que dois laços aninhados funcionam e por que um laço só nunca funcionaria.

In [ ]:
POT_MAPA = "gauss"                       # o gaussiano é suave e rápido: bom mapa
n_grade = 60
eixo_intensidade = np.linspace(0.3, 6.0, n_grade)
eixo_escala      = np.linspace(0.25, 2.2, n_grade)

mapa_inv_a = np.zeros((n_grade, n_grade))
mapa_r0    = np.zeros((n_grade, n_grade))
for i, escala_inversa in enumerate(eixo_escala):          # linhas = escala
    for j, intensidade in enumerate(eixo_intensidade):    # colunas = intensidade
        a, r0, _ = medir(FABRICAS[POT_MAPA](intensidade, escala_inversa),
                         passo_radial=5e-3)
        mapa_inv_a[i, j] = 1.0/a
        mapa_r0[i, j]    = r0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))

# --- mapa de 1/a com o contorno da unitariedade em destaque -----------------
lim = np.nanpercentile(np.abs(mapa_inv_a), 96)
im = ax1.pcolormesh(eixo_intensidade, eixo_escala, mapa_inv_a,
                    cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
plt.colorbar(im, ax=ax1, label="1/a  (fm⁻¹)")
ax1.contour(eixo_intensidade, eixo_escala, mapa_inv_a, levels=[0.0],
            colors="k", linewidths=2.2)
ax1.set(xlabel="intensidade v", ylabel="escala inversa μ (fm⁻¹)",
        title=f"{NOME_LONGO[POT_MAPA]}: mapa de 1/a\n"
              "linha preta = UNITARIEDADE (1/a = 0)")

# --- as duas famílias de curvas e o cruzamento ------------------------------
alvo_d = TABELA1["deuteron"]
c1 = ax2.contour(eixo_intensidade, eixo_escala, mapa_inv_a,
                 levels=[1.0/alvo_d["a"]], colors="#d62728", linewidths=2.2)
c2 = ax2.contour(eixo_intensidade, eixo_escala, mapa_r0,
                 levels=[alvo_d["r0"]], colors="#1f77b4", linewidths=2.2)
s = resultado_deuteron[POT_MAPA]["sintonia"]
ax2.scatter([s.p1], [s.p2], s=150, marker="*", color="k", zorder=6,
            label=f"solução da sintonia\nv={s.p1:.4f}, μ={s.p2:.4f}")
ax2.plot([], [], color="#d62728", lw=2.2, label=f"1/a = 1/{alvo_d['a']} fm⁻¹")
ax2.plot([], [], color="#1f77b4", lw=2.2, label=f"r₀ = {alvo_d['r0']} fm")
ax2.set(xlabel="intensidade v", ylabel="escala inversa μ (fm⁻¹)",
        title="A sintonia é um CRUZAMENTO de curvas de nível\n"
              "(alvo: dêuteron)")
ax2.legend(fontsize=7.5, loc="upper right")
plt.show()

### Por que dois laços, e não um

As duas curvas do painel direito são **quase paralelas** numa boa parte do
plano. Um método que tentasse resolver as duas equações de uma vez (Newton 2D)
encontraria uma matriz jacobiana mal-condicionada exatamente aí, e daria
passos gigantescos na direção errada.

Os laços aninhados evitam isso resolvendo **uma equação de cada vez**, sempre
em uma variável, sempre com bisseção — que não tem como divergir desde que o
sinal esteja encapsulado. É mais lento por iteração e infinitamente mais
confiável. Para um caderno que vai ser rodado por outra pessoa, essa troca
está certa.

---
# 6. Tabela consolidada

Tudo que este caderno produziu, num lugar só.

In [ ]:
print("=" * 104)
print(" SINTONIA AUTOMÁTICA — 12 ajustes independentes (Tabelas 3 e 4 do artigo)")
print("=" * 104)
print(f"{'caso':<16}{'potencial':<26}{'p1':>12}{'p2':>13}{'a obtido':>13}"
      f"{'r0 obtido':>11}{'nós':>5}{'conv.':>7}")
print("-" * 104)
for caso in CASOS:
    for nome in ("poco", "mpt", "gauss", "lj"):
        r = sintonias[(caso, nome)]
        print(f"{ROTULO_CASO[caso]:<16}{NOME_LONGO[nome]:<26}{r.p1:>12.6f}"
              f"{r.p2:>13.6f}{r.a:>13.5g}{r.r0:>11.4f}{r.nos:>5}"
              f"{'sim' if r.convergiu else 'NÃO':>7}")

print("\n" + "=" * 104)
print(" ESTADO LIGADO — mesma (a, r0), quatro potenciais, dois sistemas reais")
print("=" * 104)
print(f"{'sistema':<14}{'|a|/r0':>8}   {'método':<28}{'energia':>16}{'desvio':>11}")
print("-" * 104)
for rot, res in (("dêuteron", resultado_deuteron), ("dímero ⁴He", resultado_helio)):
    m = res["_meta"]; E_exp = m["alvo"]["E_ref"]; u = m["unidade"]
    razao = abs(m["alvo"]["a"])/m["alvo"]["r0"]
    print(f"{rot:<14}{razao:>8.1f}   {'EXPERIMENTO':<28}{E_exp:>13.5g} {u:<2}{'—':>10}")
    for rotulo, valor in [("fórmula alcance zero", m["E_zr"]),
                          ("fórmula alcance finito", m["E_fr"])] + \
                         [(NOME_LONGO[n], res[n]["E"]) for n in
                          ("poco", "mpt", "gauss", "lj") if n in res]:
        print(f"{'':<14}{'':>8}   {rotulo:<28}{valor:>13.5g} {u:<2}"
              f"{100*(valor/E_exp-1):>+10.2f}%")
    print("-" * 104)

---
# 7. O que este caderno estabelece

### 1. A sintonia automática funciona para qualquer potencial
Doze ajustes independentes (3 alvos × 4 formas funcionais) reproduzem os
parâmetros publicados. Os desvios ficam abaixo de 0.2% nos potenciais suaves.
O Lennard-Jones fica em 0.9–2.7%, e a causa é conhecida e escolhida: usamos
passo radial 10× mais grosso nele porque o alcance numérico da cauda $1/r^6$
chega a centenas de fm.

### 2. Dois números descrevem a física de baixa energia
Quatro potenciais sem nada em comum — um descontínuo, um suave, um gaussiano,
um com caroço duro — sintonizados no mesmo $(a, r_0)$ preveem a mesma energia
de ligação dentro de **1%**, em dois sistemas separados por nove ordens de
grandeza em energia.

### 3. O alcance efetivo não é um detalhe
| Sistema | $\lvert a\rvert/r_0$ | erro do alcance zero | erro do alcance finito |
|---|---|---|---|
| dêuteron | 3.1 | −36% | −0.05% |
| dímero de ⁴He | 11.3 | −8.6% | +0.5% |

A razão $\lvert a\rvert/r_0$ é o parâmetro de controle. **O dêuteron, apesar de
ser o exemplo didático padrão de estado ligado raso, não é universal.** O
critério usual do laboratório ($\lvert a\rvert/r_0 > 10$) o reprova.

### 4. A estrutura do espaço de parâmetros justifica o método
A varredura mostra por que a sintonia procura raiz de $1/a$ e não de $a$
(polos vs. zeros), por que a contagem de nós é obrigatória (soluções distintas
com o mesmo $(a,r_0)$), e por que dois laços aninhados batem um Newton 2D
(curvas de nível quase paralelas ⇒ jacobiana mal-condicionada).

---

## Para onde isto vai

O próximo passo é **três corpos**. A sintonia que construímos aqui é a peça
que falta lá: o trímero de Efimov precisa de um potencial de dois corpos
ancorado em $(a, r_0)$ conhecidos, e é exatamente isso que este caderno
produz. O motor de três corpos já existe em `src/tres_corpos/trimero.py`,
com a razão universal $e^{\pi/s_0} = 22.694$ verificada.

O método da dissertação — **Monte Carlo Quântico (VMC + DMC)** — vai usar
estes potenciais sintonizados como entrada.

### O que fica em aberto neste caderno

- **Sem barras de erro.** Nenhum número aqui tem incerteza de discretização
  estimada. O laboratório tem `src/comum/incerteza.py` (Richardson + portões
  de validade) pronto para isso; foi deixado de fora de propósito para manter
  o caderno legível.
- **O Lennard-Jones no dímero de hélio** não entra na tabela: com $a = 90$ Å,
  o alcance numérico da cauda vai a milhares de Å e o custo explode. Requer
  grade não-uniforme.
- **Só onda-s.** Válido enquanto $kR \ll 1$.